**Week 4 — Momentum & Collisions**

Newtons law was actualy stated in terms of momentum where the force was equal to the change of momentum over time, dp/dt. p = mv. F = dp/dy. when mass was constant dp/dt = m dv/dt = ma. The momentum version works even when mass changes   

The ky idea that is used in this week is the conservation of momentum. In an isolated system total momentum is conserved.



**One-Dimensional Collisions**

***The elastic case***

In [1]:
import numpy as np

def elastic_1d(m1, m2, u1, u2):
    """Return post-collision velocities for a 1D elastic collision."""
    v1 = ((m1 - m2) * u1 + 2 * m2 * u2) / (m1 + m2)
    v2 = ((m2 - m1) * u2 + 2 * m1 * u1) / (m1 + m2)
    return v1, v2

# Classic check: equal masses exchange velocities
print(elastic_1d(1.0, 1.0, 3.0, 0.0))   # -> (0.0, 3.0)

(0.0, 3.0)


***Three limiting cases worth memorizing***: equal masses exchange velocities; a light object hitting a massive wall reverses (v → −u); a massive object hitting a light one barely slows. If your code reproduces all three, it's almost certainly correct.

***The general case: coefficient of restitution***

In [2]:
def collide_1d(m1, m2, u1, u2, e=1.0):
    """1D collision with restitution e. e=1 elastic, e=0 perfectly inelastic."""
    total_m = m1 + m2
    # center-of-mass velocity is unchanged by the collision
    v_cm = (m1 * u1 + m2 * u2) / total_m
    v1 = v_cm - e * (m2 / total_m) * (u1 - u2)
    v2 = v_cm + e * (m1 / total_m) * (u1 - u2)
    return v1, v2

In [7]:
collide_1d(2,3,4,0,0)

(1.6, 1.6)

In [8]:

# Initial conditions
m1, m2 = 2.0, 3.0
u1, u2 = 4.0, 0.0

# Before collision metrics
p_before = (m1 * u1) + (m2 * u2)
ke_before = (0.5 * m1 * u1**2) + (0.5 * m2 * u2**2)

print(f"--- Before Collision ---")
print(f"Total Momentum: {p_before:.2f} kg·m/s")
print(f"Total KE:       {ke_before:.2f} J\n")

# Evaluate for each coefficient of restitution
for e in [1.0, 0.5, 0.0]:
    v1, v2 = collide_1d(m1, m2, u1, u2, e)
    
    # After collision metrics
    p_after = (m1 * v1) + (m2 * v2)
    ke_after = (0.5 * m1 * v1**2) + (0.5 * m2 * v2**2)
    
    print(f"--- After Collision (e = {e}) ---")
    print(f"Velocities:     v1 = {v1:+.2f} m/s, v2 = {v2:+.2f} m/s")
    print(f"Total Momentum: {p_after:.2f} kg·m/s (Conserved!)")
    print(f"Total KE:       {ke_after:.2f} J")


--- Before Collision ---
Total Momentum: 8.00 kg·m/s
Total KE:       16.00 J

--- After Collision (e = 1.0) ---
Velocities:     v1 = -0.80 m/s, v2 = +3.20 m/s
Total Momentum: 8.00 kg·m/s (Conserved!)
Total KE:       16.00 J
--- After Collision (e = 0.5) ---
Velocities:     v1 = +0.40 m/s, v2 = +2.40 m/s
Total Momentum: 8.00 kg·m/s (Conserved!)
Total KE:       8.80 J
--- After Collision (e = 0.0) ---
Velocities:     v1 = +1.60 m/s, v2 = +1.60 m/s
Total Momentum: 8.00 kg·m/s (Conserved!)
Total KE:       6.40 J


***Two-Dimensional Collisions***

In [9]:
def collide_2d(m1, m2, r1, r2, v1, v2, e=1.0):
    """Resolve a 2D collision between two circles. Positions r, velocities v are 2-vectors."""
    r1, r2 = np.asarray(r1, float), np.asarray(r2, float)
    v1, v2 = np.asarray(v1, float), np.asarray(v2, float)

    n = r2 - r1
    dist = np.linalg.norm(n)
    if dist == 0:
        return v1, v2            # degenerate; skip
    n = n / dist                 # unit normal

    # normal (scalar) components along n
    u1n, u2n = np.dot(v1, n), np.dot(v2, n)
    # tangential vector components (unchanged)
    v1t = v1 - u1n * n
    v2t = v2 - u2n * n

    # 1D collision on the normal components
    new1n, new2n = collide_1d(m1, m2, u1n, u2n, e)

    return v1t + new1n * n, v2t + new2n * n

***Detection vs. Response**

In [10]:
def overlapping(r1, r2, radius1, radius2):
    return np.linalg.norm(np.asarray(r2) - np.asarray(r1)) < (radius1 + radius2)

In [17]:
# 1. Define all necessary initial conditions first
m1, m2 = 2.0, 3.0
radius1, radius2 = 0.5, 0.5   # These replace R1 and R2
e = 1.0

# Setup positions where they are overlapping
r1 = np.array([0.0, 0.0])
r2 = np.array([0.8, 0.0])     # Distance is 0.8, which is < (0.5 + 0.5)

# Setup velocities where they are moving toward each other (approaching)
v1 = np.array([2.0, 0.0])
v2 = np.array([-1.0, 0.0])

# 2. Run the collision detection and response check
approaching = np.dot(v2 - v1, r2 - r1) < 0

# Fix: Use lowercase radius1 and radius2 here to match your variables
if overlapping(r1, r2, radius1, radius2) and approaching:
    v1, v2 = collide_2d(m1, m2, r1, r2, v1, v2, e)
    print("Collision resolved!")
    print(f"New v1: {v1}, New v2: {v2}")
else:
    print("No collision resolved.")


Collision resolved!
New v1: [-1.6  0. ], New v2: [1.4 0. ]


In [18]:
def total_momentum(bodies):
    return sum(b['m'] * b['v'] for b in bodies)   # vector sum

def total_ke(bodies):
    return sum(0.5 * b['m'] * np.dot(b['v'], b['v']) for b in bodies)